In [2]:
!pip3 install requests
!pip3 install time

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement time (from versions: none)
ERROR: No matching distribution found for time

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
import time

#A funtion to fetch airport information through ICAO code off all Airports

def fetch_arpt_info(icao_code):
	url = f"https://aerodatabox.p.rapidapi.com/airports/icao/{icao_code}" 
	headers = {
				"x-rapidapi-key": "b1394efd78msh88875d74bd69d3ap1167eajsnade9c84b7ff5",
				"x-rapidapi-host": "aerodatabox.p.rapidapi.com"
			  }
	arpt_res = requests.get(url, headers=headers)
	time.sleep(5)  # wait for 5 seconds before sending another request
	return arpt_res.json()

In [ ]:
#Creating my airport list and fetching all airports information using a for loop

my_arpt_list = ['VOMM','VOBL','VOGO','VIDP','VABB','VOHS','VOCB','OIIE','LFPG','OMDB','VRMM','RJTT','WMKL','EGLL','VOPB']

all_arpts = {}  # 15 Airports selected

for icao in my_arpt_list:
    try:
        airport_info = fetch_arpt_info(icao)
        all_arpts[icao] = airport_info
    except Exception as e:
        print(f"Error loading airport {icao}: {e}")


In [ ]:
# Defining a function to extract only specific (required) information of the aiport 

def extract_airport_info(airport_info):
    extracted_info = {
        'icao_code' : airport_info.get('icao'),
        'iata_code' : airport_info.get('iata'),
        'name' : airport_info.get('fullName'),
        'city' : airport_info.get('shortName'),
        'country' : airport_info.get('country',{}).get('name'),
        'continent' : airport_info.get('continent',{}).get('name'),
        'latitude' : airport_info.get('location',{}).get('lat'),
        'longitude' : airport_info.get('location',{}).get('lon'),
        'timezone' : airport_info.get('timeZone')
    }
    return extracted_info

In [116]:
# Adding all the 15 airports info to a list 
extracted_airports_data = []
for data in all_arpts.values():
    extracted_airports_data.append(extract_airport_info(data))

# Creating a pandas DataFrame with all the 15 airports information
import pandas as pd
airport_df = pd.DataFrame(extracted_airports_data)
display(airport_df)

,icao_code,iata_code,name,city,country,continent,latitude,longitude,timezone
0,VOMM,MAA,Chennai,Chennai,India,Asia,12.990005,80.169300,Asia/Kolkata
1,VOBL,BLR,Bangalore Bengaluru,Bengaluru,India,Asia,13.197899,77.706300,Asia/Kolkata
2,VOGO,GOI,Vasco da Gama Dabolim,Dabolim,India,Asia,15.380800,73.831400,Asia/Kolkata
3,VIDP,DEL,New Delhi Indira Gandhi,Indira Gandhi,India,Asia,28.566500,77.103100,Asia/Kolkata
4,VABB,BOM,Mumbai Chhatrapati Shivaji,Chhatrapati Shivaji,India,Asia,19.088700,72.867900,Asia/Kolkata
5,VOHS,HYD,Hyderabad Rajiv Gandhi,Rajiv Gandhi,India,Asia,17.231318,78.429855,Asia/Kolkata
6,VOCB,CJB,Coimbatore,Coimbatore,India,Asia,11.029999,77.043400,Asia/Kolkata
7,OIIE,IKA,Tehran Imam Khomeini,Imam Khomeini,Iran,Asia,35.416100,51.152200,Asia/Tehran
8,LFPG,CDG,Paris Charles de Gaulle,Charles de Gaulle,France,Europe,49.012800,2.549999,Europe/Paris
9,OMDB,DXB,Dubai,Dubai,United Arab Emirates,Asia,25.252798,55.364400,Asia/Dubai


In [ ]:
#Storing airport dataframe
%store airport_df

Stored 'airport_df' (DataFrame)


## Fetching flights information

In [ ]:
import requests
import time

# Defining a function to fetch flights information by passing icao codes of all airports
def fetch_flights_info(icao):
	url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/2026-05-20T20:00/2026-05-21T08:00"
	
	querystring = {"withLeg":"true","direction":"Both","withCancelled":"true","withCodeshared":"true","withCargo":"true","withPrivate":"true","withLocation":"false"}
	headers = {
		"x-rapidapi-key": "b1394efd78msh88875d74bd69d3ap1167eajsnade9c84b7ff5",
		"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
		"Content-Type": "application/json"
				}

	flight_response = requests.get(url, headers=headers, params=querystring)
	time.sleep(5)
	
	return flight_response.json()

In [ ]:
# Extracting flights information

def extract_flight_info(flight_data, flight_type, airport_icao):
    flight_id = f"{flight_data.get('number', '')}_{flight_data.get(flight_type, {}).get('scheduledTime', {}).get('utc', '')}"

    # Determine origin and destination based on flight type, ensuring ICAO codes
    origin_code = ''
    destination_code = ''
    if flight_type == 'departure':
        origin_code = airport_icao
        destination_code = flight_data.get('arrival',{}).get('airport',{}).get('icao')
    elif flight_type == 'arrival':
        origin_code = flight_data.get('departure',{}).get('airport',{}).get('icao')
        destination_code = airport_icao

    return {
        'flight_id' : flight_id,
        'flight_number' : flight_data.get('number', ''),
        'aircraft_registration' : flight_data.get('aircraft', {}).get('reg'),
        'origin_icao' : origin_code,
        'destination_code' : destination_code,
        'scheduled_departure' : flight_data.get('departure', {}).get('scheduledTime', {}).get('local'),
        'actual_departure' : flight_data.get('departure', {}).get('revisedTime', {}).get('local', flight_data.get('departure', {}).get('scheduledTime', {}).get('local')),
        'scheduled_arrival' : flight_data.get('arrival', {}).get('scheduledTime', {}).get('local'),
        'actual_arrival' : flight_data.get('arrival', {}).get('revisedTime', {}).get('local', flight_data.get('arrival', {}).get('scheduledTime', {}).get('local')),
        'status' : flight_data.get('status'),
        'airline_code' : flight_data.get('airline').get('iata')
    }

In [ ]:
# Limiting the flights for each airport 
# Fetching all flights information using a for loop

flight_limit = {
    'VOMM':120,'VOBL':125,'VOGO':118,'VIDP':115,'VABB':138,   
    'VOHS':152,'VOCB':147,'OIIE':117,'LFPG':111,'OMDB':115,
    'VRMM':180,'RJTT':170,'WMKL':130,'EGLL':140,'VOPB':126
}

all_flights_data = []  # 1292

for icao,limit in flight_limit.items():
    try:
        flight_api_response = fetch_flights_info(icao)
        #applying limit as per mentioned
        limit = flight_limit.get(icao, 100)
        
        #Extract departures
        if 'departures' in flight_api_response:
            for flight in flight_api_response['departures'][:limit//2]:
                all_flights_data.append(extract_flight_info(flight, 'departure', icao))
        
        #Extract arrivals
        if 'arrivals' in flight_api_response:
            for flight in flight_api_response['arrivals'][:limit//2]:
                all_flights_data.append(extract_flight_info(flight, 'arrival', icao))
    except Exception as e:
        print(f"Error loading flight {icao} : {e}")

In [243]:
# A DataFrame with all flights information 1590
import pandas as pd

flights_df = pd.DataFrame(all_flights_data)
display(flights_df)

,flight_id,flight_number,aircraft_registration,origin_icao,destination_code,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,status,airline_code
0,6E 7592_2026-05-20 14:30Z,6E 7592,NaN,VOMM,VOMD,2026-05-20 20:00+05:30,2026-05-20 20:00+05:30,2026-05-20 21:25+05:30,2026-05-20 21:25+05:30,Unknown,6E
1,6E 6812_2026-05-20 14:30Z,6E 6812,NaN,VOMM,VOCB,2026-05-20 20:00+05:30,2026-05-20 20:00+05:30,2026-05-20 21:05+05:30,2026-05-20 21:05+05:30,Unknown,6E
2,6E 193_2026-05-20 14:30Z,6E 193,NaN,VOMM,VOHS,2026-05-20 20:00+05:30,2026-05-20 20:00+05:30,2026-05-20 21:25+05:30,2026-05-20 21:25+05:30,Unknown,6E
3,6E 5149_2026-05-20 14:45Z,6E 5149,VT-IWT,VOMM,VABB,2026-05-20 20:15+05:30,2026-05-20 20:15+05:30,2026-05-20 22:20+05:30,2026-05-20 22:20+05:30,Unknown,6E
4,6E 7051_2026-05-20 14:50Z,6E 7051,NaN,VOMM,VOTR,2026-05-20 20:20+05:30,2026-05-20 20:20+05:30,2026-05-20 21:30+05:30,2026-05-20 21:30+05:30,Unknown,6E
...,...,...,...,...,...,...,...,...,...,...,...
1287,LH 920_2026-05-20 19:40Z,LH 920,D-AIJM,EDDF,EGLL,2026-05-20 20:00+02:00,2026-05-20 20:00+02:00,2026-05-20 20:40+01:00,2026-05-20 20:38+01:00,Arrived,LH
1288,A3 1820_2026-05-20 19:40Z,A3 1820,D-AIJM,EDDF,EGLL,2026-05-20 20:00+02:00,2026-05-20 20:00+02:00,2026-05-20 20:40+01:00,2026-05-20 20:38+01:00,Arrived,A3
1289,BA 1373_2026-05-20 19:50Z,BA 1373,NaN,EGCC,EGLL,2026-05-20 19:50+01:00,2026-05-20 19:54+01:00,2026-05-20 20:50+01:00,2026-05-20 20:40+01:00,Arrived,BA
1290,6E 581_2026-05-21 01:55Z,6E 581,NaN,VOPB,VOMM,2026-05-21 07:25+05:30,2026-05-21 07:25+05:30,2026-05-21 09:35+05:30,2026-05-21 09:35+05:30,Unknown,6E


In [ ]:
# Storing fligths dataframe
%store flights_df

Stored 'flights_df' (DataFrame)


### Fetching Aircraft Information

In [ ]:
#Adding aircraft registration numbers as available in the flights_df to a list
all_aircraft_reg = []   #499

# Considering aircraft_registration column without null values
arcrft_reg = flights_df.loc[flights_df['aircraft_registration'].notna(), 'aircraft_registration']     # 793

'''df.loc[condition, column_name]   -> Takes rows satisfying condition & extract only specified column
.loc[] is a Pandas label-based indexing method used to select rows and columns by labels (names) or conditions.
                          dataframe.loc[row_selection, column_selection

df.iloc[]  -> integer location
It is used to select rows and columns by integer position (index number), not by labels or conditions.
'''

# Appeding unique aircraft registrations to all_aircraft_reg
for reg in arcrft_reg:
    if reg not in all_aircraft_reg:
        all_aircraft_reg.append(reg)

In [ ]:
# A function to fetch aircraft information of all aircrafts

import requests
import time

aircraft_no_info = []   # 16 aircrafts


def fetch_aircraft_info(reg_no):
    url = f"https://aerodatabox.p.rapidapi.com/aircrafts/reg/{reg_no}/all"

    headers = {
		"x-rapidapi-key": "b1394efd78msh88875d74bd69d3ap1167eajsnade9c84b7ff5",
		"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	    "Content-Type": "application/json"
            }
        
    aircraft_response = requests.get(url, headers=headers)
    time.sleep(1)

    status = aircraft_response.status_code

    #Checking if all my aircrafts have information
    if status == 204: 
        aircraft_no_info.append(reg_no)
        return None
    elif status == 200:
        return aircraft_response.json()
    else:
        print(f"{reg_no} -> Status Code: {status}")
        return None
        

In [ ]:
#Creating my airport list and fetching all airports information using a for loop

failed_aircrafts = []  #NA
all_aircrafts = {}     #483 

for reg in all_aircraft_reg:
    try:
        aircraft_info = fetch_aircraft_info(reg)
        if aircraft_info is not None:
            all_aircrafts[reg] = aircraft_info
    except Exception as e:   
        failed_aircrafts.append(reg)
        print(f"Error loading airport {reg}: {e}")

In [126]:
%store aircraft_no_info

Stored 'aircraft_no_info' (list)


In [100]:
# Defining a function to extract only specific (required) information of the aircraft

def extract_aircraft_info(airc_info):
    if type(airc_info) is list:
        aircraft_info = airc_info[0]
    else:
        aircraft_info = airc_info

    aircraft_extract ={
        'registration' : aircraft_info.get('reg'),
        'model' : f"{aircraft_info.get('model')} {aircraft_info.get('modelCode')}",
        'manufacturer' : (aircraft_info.get('productionLine').split()[0] if aircraft_info.get('productionLine') else None),
        'icao_type_code' : aircraft_info.get('icaoCode'),
        'owner' : aircraft_info.get('airlineName')
    }
    return aircraft_extract

In [71]:
# Adding all the extracted aircrafts information to a list

extracted_aircrafts_data = []
for info in all_aircrafts.values():
        extracted_aircrafts_data.append(extract_aircraft_info(info))

# Creating a DF
import pandas as pd
aircraft_df = pd.DataFrame(extracted_aircrafts_data)
display(aircraft_df)

,registration,model,manufacturer,icao_type_code,owner
0,VT-IWT,A21N 321-252NX,Airbus,A21N,IndiGo
1,VT-ALS,B773 B777-337ER,Boeing,B773,Air India
2,VT-IBD,A21N 321-252NX,Airbus,A21N,IndiGo
3,VT-IBC,A21N 321-252NX,Airbus,A21N,IndiGo
4,VT-BOM,A320 320-214,Airbus,A320,Air India Express
...,...,...,...,...,...
478,G-TTSI,A320 320-251,Airbus,A320,British Airways
479,G-TNEH,A21N 321-251NX,Airbus,A21N,British Airways
480,PH-AXK,A21N 321-252NX,Airbus,A21N,KLM
481,G-NEOU,A21N 321-251NX,Airbus,A321,British Airways


In [ ]:
%store aircraft_df
%store -r airport_df

Stored 'aircraft_df' (DataFrame)


### Fetching airport delays information

In [73]:
import requests
import time

arpts_no_delay_info = []   # 6 airports with no delay info -> ['VOGO', 'VOHS', 'VOCB', 'OIIE', 'VRMM', 'VOPB']

def fetch_arpt_delays_info(icao):
    url = f"https://aerodatabox.p.rapidapi.com/airports/icao/{icao}/delays/2026-05-21T12:00/2026-05-22T00:00"

    headers = {
	    "x-rapidapi-key": "86738f447dmsh8b41e537a97e210p1ea4b3jsn3d4dd8302442",
	    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	    "Content-Type": "application/json"
            }
    
    delay_response = requests.get(url, headers=headers)
    time.sleep(1)

    status = delay_response.status_code

    #Checking if all my airports have delay information on the date specified
    if status == 204:  # status code 204 means no content
        arpts_no_delay_info.append(icao)
        return None
    elif status == 200:
        return delay_response.json()
    else:   
        print(f"{icao} -> Status Code: {status}")
        return None

In [ ]:
my_arpt_list = ['VOMM','VOBL','VOGO','VIDP','VABB','VOHS','VOCB','OIIE','LFPG','OMDB','VRMM','RJTT','WMKL','EGLL','VOPB']

failed_records = []  # NA
all_airport_delays = {}   # 9

for icao_code in my_arpt_list:
    try:
        info = fetch_arpt_delays_info(icao_code)
        if info is not None:
            delay_info = info[0]
            all_airport_delays[icao_code] = delay_info  
    except Exception as e:
        failed_records.append(icao_code)
        print(f"Error loading airport delays {icao_code}: {e}")

### Calculating Delays

In [94]:
## Departure delays -> Converting it to datetime format
flights_df['scheduled_departure'] = pd.to_datetime(flights_df['scheduled_departure'], utc=True)
flights_df['actual_departure'] = pd.to_datetime(flights_df['actual_departure'], utc=True)

# a positive value indicates (delay) and negative value indicates (early)
flights_df['dep_delay'] = (flights_df['actual_departure'] - flights_df['scheduled_departure']).dt.total_seconds() /60  # diving by 60 to get it in minutes
''' .dt.total_seconds()  -> used to convert duration / timedelta values into a single floating point number representing the total seconds.'''

airport_departure_delay = (flights_df.groupby('origin_icao').agg(delayed_departures = ('dep_delay', lambda x:(x>0).sum()), avg_departure_delay_min = ('dep_delay', 'mean'), median_departure_delay_min = ('dep_delay', 'median')))


## Arrival delays -> Converting it to datetime format
flights_df['scheduled_arrival'] = pd.to_datetime(flights_df['scheduled_arrival'], utc=True)
flights_df['actual_arrival'] = pd.to_datetime(flights_df['actual_arrival'], utc=True)

# a positive value indicates (delay) and negative value indicates (early)
flights_df['arr_delay'] = (flights_df['actual_arrival'] - flights_df['scheduled_arrival']).dt.total_seconds() / 60

airport_arrival_delay = (flights_df.groupby('origin_icao').agg(delayed_arrivals = ('arr_delay', lambda x:(x>0).sum()), avg_arrival_delay_min = ('arr_delay', 'mean'), median_arrival_delay_min = ('arr_delay', 'median')))

In [95]:
# Extracting specified information from the 

def extract_airport_delay_info(delay):    
    icao = delay.get('airportIcao')
    
    delay_extract = {
        'airport_iata' : airport_df.loc[airport_df['icao_code'] == icao, 'iata_code'].iloc[0],
        'delay_date' : delay.get('from',{}).get('local').split()[0],
        'total_flights' : delay.get('departuresDelayInformation',{}).get('numTotal') + delay.get('arrivalsDelayInformation',{}).get('numTotal'),
        'delayed_departures' : int(airport_departure_delay.loc[icao, 'delayed_departures']),  # type casting to int as it return as numpy integer datatype (np.int64(2))
        'delayed_arrivals' : int(airport_arrival_delay.loc[icao, 'delayed_arrivals']),
        'avg_departure_delay_min' : int(airport_departure_delay.loc[icao, 'avg_departure_delay_min']),
        'avg_arrival_delay_min' : int(airport_arrival_delay.loc[icao, 'avg_arrival_delay_min']),
        'median_departure_delay_min' : int(airport_departure_delay.loc[icao, 'median_departure_delay_min']),
        'median_arrival_delay_min' : int(airport_arrival_delay.loc[icao, 'median_arrival_delay_min']),
        'canceled_flights' : delay.get('departuresDelayInformation',{}).get('numCancelled')+delay.get('arrivalsDelayInformation',{}).get('numCancelled')
    }

    return delay_extract


In [96]:
extracted_airport_delay = []

for dt in all_airport_delays.values():
    extracted_airport_delay.append(extract_airport_delay_info(dt))


# Creating a DataFrame

import pandas as pd

airport_delay_df = pd.DataFrame(extracted_airport_delay)
display(airport_delay_df)

,airport_iata,delay_date,total_flights,delayed_departures,delayed_arrivals,avg_departure_delay_min,avg_arrival_delay_min,median_departure_delay_min,median_arrival_delay_min,canceled_flights
0,MAA,2026-05-21,46,0,5,0,-3,0,0,0
1,BLR,2026-05-21,77,23,1,7,-1,0,0,0
2,DEL,2026-05-21,154,0,12,0,1,0,0,3
3,BOM,2026-05-21,88,26,21,9,4,0,0,0
4,CDG,2026-05-21,182,58,6,17,-3,13,-3,0
5,DXB,2026-05-21,67,44,2,6,-8,3,0,3
6,HND,2026-05-21,179,52,10,13,-5,4,0,2
7,LGK,2026-05-21,12,0,0,0,-9,0,-9,0
8,LHR,2026-05-21,163,72,25,36,3,20,-4,2


In [97]:
%store airport_delay_df

Stored 'airport_delay_df' (DataFrame)


In [2]:
%store -r airport_df
%store -r aircraft_df
%store -r flights_df
%store -r airport_delay_df

In [4]:
display(airport_df)

,icao_code,iata_code,name,city,country,continent,latitude,longitude,timezone
0,VOMM,MAA,Chennai,Chennai,India,Asia,12.990005,80.169300,Asia/Kolkata
1,VOBL,BLR,Bangalore Bengaluru,Bengaluru,India,Asia,13.197899,77.706300,Asia/Kolkata
2,VOGO,GOI,Vasco da Gama Dabolim,Dabolim,India,Asia,15.380800,73.831400,Asia/Kolkata
3,VIDP,DEL,New Delhi Indira Gandhi,Indira Gandhi,India,Asia,28.566500,77.103100,Asia/Kolkata
4,VABB,BOM,Mumbai Chhatrapati Shivaji,Chhatrapati Shivaji,India,Asia,19.088700,72.867900,Asia/Kolkata
5,VOHS,HYD,Hyderabad Rajiv Gandhi,Rajiv Gandhi,India,Asia,17.231318,78.429855,Asia/Kolkata
6,VOCB,CJB,Coimbatore,Coimbatore,India,Asia,11.029999,77.043400,Asia/Kolkata
7,OIIE,IKA,Tehran Imam Khomeini,Imam Khomeini,Iran,Asia,35.416100,51.152200,Asia/Tehran
8,LFPG,CDG,Paris Charles de Gaulle,Charles de Gaulle,France,Europe,49.012800,2.549999,Europe/Paris
9,OMDB,DXB,Dubai,Dubai,United Arab Emirates,Asia,25.252798,55.364400,Asia/Dubai


## Connecting to MySql

In [1]:
import mysql.connector

connection = mysql.connector.connect(
    host ="localhost",
    user="root",
    password ="1234567",
    database ='Flight_Analytics'
)
cursor = connection.cursor()
connection.commit()

#### Database Creation

In [ ]:
#Creating a database Flight_Analytics
cursor.execute('CREATE DATABASE Flight_Analytics')
#Mentioning to use the newly created database
cursor.execute('USE Flight_Analytics')

#### Tables Creation in the Flight_Analytics Database

In [ ]:
# Created the following tables

airport_table = '''
CREATE TABLE airport(
    airport_id INTEGER PRIMARY KEY AUTO_INCREMENT,
    icao_code VARCHAR(4) UNIQUE,
    iata_code VARCHAR(3) UNIQUE,
    name VARCHAR(100), 
    city VARCHAR(100), 
    country VARCHAR(100),
    continent VARCHAR(50),
    latitude DECIMAL(10,6),
    longitude DECIMAL(10,6),
    timezone VARCHAR(50)
)
'''

aircraft_table = '''
CREATE TABLE aircraft(
    aircraft_id INTEGER PRIMARY KEY AUTO_INCREMENT,
    registration VARCHAR(50) UNIQUE,
    model VARCHAR(100),
    manufacturer VARCHAR(50),
    icao_type_code VARCHAR(10),
    owner VARCHAR(100)
)
'''

flights_table = '''
CREATE TABLE flights(
    flight_no INTEGER PRIMARY KEY AUTO_INCREMENT,
    flight_id VARCHAR(100),
    flight_number VARCHAR(20),
    aircraft_registration VARCHAR(50) NULL,
    origin_icao VARCHAR(4) NULL,
    destination_code VARCHAR(4),
    scheduled_departure DATETIME,
    actual_departure DATETIME,
    scheduled_arrival DATETIME,
    actual_arrival DATETIME,
    status VARCHAR(30),
    airline_code VARCHAR(10),
    FOREIGN KEY (aircraft_registration)
        REFERENCES aircraft(registration),
    FOREIGN KEY (origin_icao)
        REFERENCES airport(icao_code)
)
'''

airport_delays_table = '''
CREATE TABLE airport_delays(
    delay_id INTEGER PRIMARY KEY AUTO_INCREMENT,
    airport_iata VARCHAR(3),
    delay_date DATETIME,
    total_flights INTEGER,
    delayed_departures_flights INTEGER,
    delayed_arrivals_flights INTEGER,
    avg_departure_delay_min INTEGER,
    avg_arrival_delay_min INTEGER,
    median_departure_delay_min INTEGER,
    median_arrival_delay_min INTEGER,
    cancelled_flights INTEGER,
    FOREIGN KEY (airport_iata)
        REFERENCES airport(iata_code)
)
'''

cursor.execute(airport_table)
cursor.execute(aircraft_table)
cursor.execute(flights_table)
cursor.execute(airport_delays_table)   

In [10]:
# Viewing tables available in my database flight_analytics
q = 'SHOW TABLES'
cursor.execute(q)

for tab in cursor:
    print(tab)

('aircraft',)
('airport',)
('airport_delays',)
('flights',)


### Inserting values into SQL

In [230]:
# AIRPORT_DF
# Checking if my airport_df columns have any null values
airport_df.isna().sum()  

icao_code    0
iata_code    0
name         0
city         0
country      0
continent    0
latitude     0
longitude    0
timezone     0
dtype: int64

In [28]:
# Inserting airport_df into airport table in MySql

data_list = airport_df.values.tolist()
query = '''
    INSERT INTO airport (icao_code, iata_code, name, city, country, continent, latitude, longitude, timezone)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''
cursor.executemany(query, data_list)
connection.commit()
print("Data inserted using to_list()")

Data inserted using to_list()


In [ ]:
# AIRCRAFT_DF
# Checking if my aircraft_df columns have any null values
aircraft_df.isna().sum()

registration      0
model             0
manufacturer      3
icao_type_code    4
owner             0
dtype: int64

In [ ]:
aircraft_df = aircraft_df.fillna('')  # Filling the null values with string
aircraft_df.isna().sum()

registration      0
model             0
manufacturer      0
icao_type_code    0
owner             0
dtype: int64

In [ ]:
# Inserting aircraft_df into aircraft table in MySql

data_list = aircraft_df.values.tolist()

query = '''
    INSERT INTO aircraft (registration, model, manufacturer, icao_type_code, owner)
    VALUES (%s,%s,%s,%s,%s)
'''

cursor.executemany(query, data_list)
connection.commit()
print("Data inserted using to_list()")

Data inserted using to_list()


In [26]:
# FLIGTHS_DF
# Retreiving flights dataframe
# Rereiving aircraft_no_info dataframe

%store -r flights_df
%store -r aircraft_no_info          

In [27]:
#Checking my flights datatypes
flights_df.dtypes

flight_id                           str
flight_number                       str
aircraft_registration               str
origin_icao                         str
destination_code                    str
scheduled_departure      datetime64[ns]
actual_departure         datetime64[ns]
scheduled_arrival        datetime64[ns]
actual_arrival           datetime64[ns]
status                              str
airline_code                        str
dtype: object

In [28]:
print(flights_df['scheduled_departure'])  # +00:00 means UTC Timezone information is attached

0      2026-05-20 14:30:00
1      2026-05-20 14:30:00
2      2026-05-20 14:30:00
3      2026-05-20 14:45:00
4      2026-05-20 14:50:00
               ...        
1287   2026-05-20 18:00:00
1288   2026-05-20 18:00:00
1289   2026-05-20 18:50:00
1290   2026-05-21 01:55:00
1291   2026-05-20 23:10:00
Name: scheduled_departure, Length: 1292, dtype: datetime64[ns]


In [29]:
print(flights_df['scheduled_departure'].dt.tz_localize(None))  # removes the timezone information from a pandas datetime column

0      2026-05-20 14:30:00
1      2026-05-20 14:30:00
2      2026-05-20 14:30:00
3      2026-05-20 14:45:00
4      2026-05-20 14:50:00
               ...        
1287   2026-05-20 18:00:00
1288   2026-05-20 18:00:00
1289   2026-05-20 18:50:00
1290   2026-05-21 01:55:00
1291   2026-05-20 23:10:00
Name: scheduled_departure, Length: 1292, dtype: datetime64[ns]


In [30]:
# Change aircraft registrations to null that don't exist in aircraft table
import pandas as pd
my_arpt_list = ['VOMM','VOBL','VOGO','VIDP','VABB','VOHS','VOCB','OIIE','LFPG','OMDB','VRMM','RJTT','WMKL','EGLL','VOPB']

# Changing flights_df['origin_icao'] to null for those missing in my_arpt_list
missing_codes = set(flights_df['origin_icao'].dropna()) - set(my_arpt_list)

flights_df.loc[flights_df['aircraft_registration'].isin(aircraft_no_info),'aircraft_registration'] = None
flights_df.loc[flights_df['origin_icao'].isin(missing_codes),'origin_icao'] = None

# Removing timezone information from datetime_cols as MySQL DATETIME columns store in the format YYYY-MM-DD HH:MM:SS
datetime_cols = ['scheduled_departure', 'actual_departure','scheduled_arrival','actual_arrival']
for cols in datetime_cols:
    flights_df[cols] = (flights_df[cols].dt.tz_localize(None)) # removing the timezone information from the specified pandas datetime column

# Converting each column to generic Python 'object' type in flights_df
flights_df = flights_df.astype(object)

# Convert all remaining null values in flights_df to None
flights_df = flights_df.where(pd.notnull(flights_df), None)
''' pd.notnull(flights_df) -> creates a Dataframe of True/False values

df.where(condition, replacement)
    if condition is True -> keeps original value
    if condition is False -> replace with replacement
'''

' pd.notnull(flights_df) -> creates a Dataframe of True/False values\n\ndf.where(condition, replacement)\n    if condition is True -> keeps original value\n    if condition is False -> replace with replacement\n'

In [32]:
# Inserting flights_df into flights table in MySql

import pandas as pd
data_list = flights_df.values.tolist()

query = '''
    INSERT INTO flights (flight_id, flight_number, aircraft_registration, origin_icao, destination_code, scheduled_departure, actual_departure, 
    scheduled_arrival, actual_arrival, status, airline_code)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''
cursor.executemany(query, data_list)
connection.commit()

In [208]:
%store flights_df

Stored 'flights_df' (DataFrame)


In [ ]:
# AIRCRAFT_DELAY_DF
# Retrieving airport_delay_df
%store -r airport_delay_df

In [119]:
airport_delay_df.isna().sum()

airport_iata                  0
delay_date                    0
total_flights                 0
delayed_departures            0
delayed_arrivals              0
avg_departure_delay_min       0
avg_arrival_delay_min         0
median_departure_delay_min    0
median_arrival_delay_min      0
canceled_flights              0
dtype: int64

In [120]:
airport_delay_df.dtypes

airport_iata                    str
delay_date                      str
total_flights                 int64
delayed_departures            int64
delayed_arrivals              int64
avg_departure_delay_min       int64
avg_arrival_delay_min         int64
median_departure_delay_min    int64
median_arrival_delay_min      int64
canceled_flights              int64
dtype: object

In [123]:
airport_delay_df.columns

Index(['airport_iata', 'delay_date', 'total_flights', 'delayed_departures',
       'delayed_arrivals', 'avg_departure_delay_min', 'avg_arrival_delay_min',
       'median_departure_delay_min', 'median_arrival_delay_min',
       'canceled_flights'],
      dtype='str')

In [128]:
# Inserting airport_delay_df into airport_delays table in MySql

import pandas as pd

data_list = airport_delay_df.values.tolist()  #converting airport_delay_df to a list

query = '''
    INSERT INTO airport_delays (airport_iata, delay_date, total_flights, delayed_departures_flights, delayed_arrivals_flights, avg_departure_delay_min, 
    avg_arrival_delay_min, median_departure_delay_min, median_arrival_delay_min, cancelled_flights)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
'''

cursor.executemany(query, data_list)
connection.commit()

### SQL QUERIES

1. Show the total number of flights for each aircraft model, listing the model and its count.

In [ ]:
q1 = '''
    SELECT aircraft.model, COUNT(*) AS total_flights FROM flights
    JOIN aircraft ON flights.aircraft_registration = aircraft.registration
    GROUP BY aircraft.model
    ORDER BY total_flights DESC
'''
cursor.execute(q1)

for info in cursor:
    print(info)

('A20N 320-251N', 123)
('A21N 321-252NX', 102)
('A21N 321-251NX', 57)
('B789 B787-9', 43)
('B38M 737-8MAX', 41)
('A320 320-214', 38)
('A359 350-941', 30)
('A20N 320-271N', 23)
('AT75 72-600', 21)
('BCS3 220-371', 19)
('A320 320-232', 18)
('B773 B777-31HER', 14)
('B738 B737-846', 13)
('B38M 737-8MAX 200', 12)
('A320 320-216', 9)
('None None', 9)
('A319 319-131', 9)
('A332 330-243', 9)
('B781 B787-10', 8)
('B737 B737-781', 8)
('A333 330-343X', 7)
('B738 B737-881', 6)
('A388 380-861', 6)
('A388 380-842', 6)
('A388 380-841', 6)
('B773 B777-381ER', 6)
('A333 330-323X', 5)
('B763 B767-322ER', 5)
('E190 190STD', 5)
('B738 B737-86N', 5)
('B763 B767-346ER', 5)
('B781 787-10', 5)
('B788 787-8', 5)
('E190 190LR', 4)
('B738 737-86N', 4)
('A21N 321-272N', 4)
('A333 330-303', 4)
('A21N 321-271NX', 3)
('B789 787-9', 3)
('A321 321-231', 3)
('B763 B767-381ER', 3)
('B738 B737-8MAX', 2)
('B738 B737-89P', 2)
('A21N 321-253NX', 2)
('B738 B737-8HJ', 2)
('A321 321-211', 2)
('B772 B777-21HLR', 2)
('A332 330-2

2. List all aircraft (registration, model) that have been assigned to more than 5 flights.

In [7]:
q2 = '''
    SELECT
        aircraft.registration,
        aircraft.model,
        COUNT(*) AS total_flights
    FROM aircraft
    JOIN flights
        ON aircraft.registration = flights.aircraft_registration
    GROUP BY aircraft.registration,
             aircraft.model
    HAVING COUNT(*) > 5
    ORDER BY total_flights DESC
'''
cursor.execute(q2)

for info in cursor:
    print(info)


('F-HPNZ', 'BCS3 220-371', 18)
('F-GKXN', 'A320 320-214', 16)
('JA878J', 'B789 B787-9', 14)
('PH-AXK', 'A21N 321-252NX', 14)
('F-GKXS', 'A320 320-214', 10)
('F-HBLN', 'E190 190STD', 10)
('JA879J', 'B789 B787-9', 10)
('G-DBCH', 'A319 319-131', 10)
('G-ZBLC', 'B781 B787-10', 10)
('N661UA', 'B763 B767-322ER', 10)
('VT-ILE', 'A21N 321-252NX', 8)
('F-HTYQ', 'A359 350-941', 8)
('VT-IBM', 'A21N 321-252NX', 8)
('VT-NCD', 'A21N 321-252NX', 8)
('D-AIJM', 'A20N 320-271N', 8)
('VT-CID', 'A20N 320-251N', 8)
('VT-ILY', 'A21N 321-252NX', 8)
('F-HPNE', 'BCS3 220-371', 8)
('VT-ILL', 'A21N 321-252NX', 8)
('F-HBLD', 'E190 190LR', 8)
('N383HA', 'A332 330-243', 8)
('JA796A', 'B773 B777-381ER', 8)
('G-NEOP', 'A21N 321-251NX', 8)
('G-XLEL', 'A388 380-841', 8)
('G-TTNH', 'A20N 320-251N', 8)
('G-TNEH', 'A21N 321-251NX', 8)
('A6-EER', 'A388 380-861', 6)
('N396HA', 'A332 330-243', 6)
('VT-IBD', 'A21N 321-252NX', 6)
('JA922A', 'B789 B787-9', 6)
('VT-IIA', 'A20N 320-251N', 6)
('JA899A', 'B789 B787-9', 6)
('VT-BWK'

3. For each airport, display its name and the number of outbound flights, but only for airports with more than 5 flights.

In [39]:
q3 = '''
    SELECT name, COUNT(flights.flight_id) AS no_of_outbound_flights FROM airport
    JOIN flights ON airport.icao_code = flights.origin_icao
    GROUP BY name
    HAVING no_of_outbound_flights > 5
    ORDER BY no_of_outbound_flights DESC
'''
cursor.execute(q3)

for info in cursor:
    print(info)

('Mumbai Chhatrapati Shivaji', 101)
('New Delhi Indira Gandhi', 91)
('Hyderabad Rajiv Gandhi', 87)
('Bangalore Bengaluru', 87)
('Tokyo Haneda', 85)
('Chennai', 72)
('London Heathrow', 72)
('Dubai', 65)
('Paris Charles de Gaulle', 58)
('Vasco da Gama Dabolim', 24)
('Coimbatore', 19)
('Malé', 18)


4. Find the top 3 destination airports (name, city) by number of arriving flights, sorted by count descending.

In [44]:
q4 = '''
    SELECT name, city, COUNT(flights.flight_id) as no_of_arriving_flights FROM airport
    JOIN flights on airport.icao_code = flights.destination_code
    GROUP BY name, city
    ORDER BY no_of_arriving_flights DESC
    LIMIT 3
'''
cursor.execute(q4)

for info in cursor:
    print(info)

('Mumbai Chhatrapati Shivaji', 'Chhatrapati Shivaji', 218)
('Hyderabad Rajiv Gandhi', 'Rajiv Gandhi', 196)
('New Delhi Indira Gandhi', 'Indira Gandhi', 190)


5. Show for each flight: number, origin, destination, and a label 'Domestic' or 'International' using CASE WHEN on country match.

In [52]:
q5 = '''
    SELECT flight_number, origin_icao, destination_code,
        CASE 
            WHEN a1.country = a2.country THEN 'Domestic'
            ELSE 'International'
        END AS flight_type
    FROM flights
    JOIN airport a1 ON flights.origin_icao = a1.icao_code
    JOIN airport a2 ON flights.destination_code = a2.icao_code
'''
cursor.execute(q5)

for info in cursor:
    print(info)

('6E 6812', 'VOMM', 'VOCB', 'Domestic')
('6E 193', 'VOMM', 'VOHS', 'Domestic')
('6E 5149', 'VOMM', 'VABB', 'Domestic')
('AI 538', 'VOMM', 'VIDP', 'Domestic')
('AI 436', 'VOMM', 'VABB', 'Domestic')
('6E 731', 'VOMM', 'VOCB', 'Domestic')
('6E 6487', 'VOMM', 'VOHS', 'Domestic')
('IX 2662', 'VOMM', 'VOBL', 'Domestic')
('AI 2838', 'VOMM', 'VIDP', 'Domestic')
('6E 796', 'VOMM', 'VOBL', 'Domestic')
('EK 547', 'VOMM', 'OMDB', 'International')
('6E 951', 'VOMM', 'VIDP', 'Domestic')
('6E 1608', 'VOMM', 'VABB', 'Domestic')
('6E 6225', 'VOMM', 'VABB', 'Domestic')
('6E 341', 'VOMM', 'VOHS', 'Domestic')
('6E 2369', 'VOMM', 'VIDP', 'Domestic')
('6E 6140', 'VOMM', 'VABB', 'Domestic')
('6E 953', 'VOMM', 'VIDP', 'Domestic')
('EK 543', 'VOMM', 'OMDB', 'International')
('6E 526', 'VOMM', 'VOPB', 'Domestic')
('6E 5093', 'VOMM', 'VABB', 'Domestic')
('BA 36', 'VOMM', 'EGLL', 'International')
('6E 189', 'VOMM', 'VOBL', 'Domestic')
('6E 613', 'VOMM', 'VIDP', 'Domestic')
('6E 5367', 'VOMM', 'VABB', 'Domestic')


6. Show the 5 most recent arrivals at “DEL” airport including flight number, aircraft, departure airport name, and arrival time, ordered by latest arrival.

In [12]:
q6 = '''
    SELECT flight_number, aircraft_registration, airport.name AS departure_airport_name, actual_arrival AS arrival_time FROM flights
    JOIN airport ON flights.origin_icao = airport.icao_code
    WHERE flights.destination_code = (
        SELECT icao_code FROM airport 
        WHERE iata_code = "DEL")
    ORDER BY arrival_time DESC
    LIMIT 5
'''
cursor.execute(q6)

for info in cursor:
    print(info)

('QP 1405', None, 'Hyderabad Rajiv Gandhi', datetime.datetime(2026, 5, 21, 2, 25))
('AI 1806', None, 'Hyderabad Rajiv Gandhi', datetime.datetime(2026, 5, 21, 1, 47))
('6E 709', None, 'Hyderabad Rajiv Gandhi', datetime.datetime(2026, 5, 21, 1, 46))
('6E 953', 'VT-NCE', 'Chennai', datetime.datetime(2026, 5, 20, 23, 36))
('6E 706', None, 'Hyderabad Rajiv Gandhi', datetime.datetime(2026, 5, 20, 22, 53))


7. Find all airports with no arriving flights (never used as a destination in flights table).

In [ ]:
q7 = '''
    SELECT airport.icao_code, airport.name, airport.city FROM airport
    LEFT JOIN flights ON airport.icao_code = flights.destination_code
    WHERE flights.destination_code IS NULL;
'''
cursor.execute(q7)

for info in cursor:
    print(info)      # Returns no rows as all airports in the current dataset have at least one arriving flight.

In [ ]:
# Checking if all my airports are available in flights destination

q = '''
    SELECT icao_code FROM airport
    WHERE icao_code IN(
        SELECT destination_code FROM flights
        WHERE destination_code IS NOT NULL
    )
'''
cursor.execute(q)

for info in cursor:
    print(info)

('EGLL',)
('LFPG',)
('OIIE',)
('OMDB',)
('RJTT',)
('VABB',)
('VIDP',)
('VOBL',)
('VOCB',)
('VOGO',)
('VOHS',)
('VOMM',)
('VOPB',)
('VRMM',)
('WMKL',)


In [ ]:
# Checking how many times each of my airport appear in flights.destination 

q = '''
    SELECT DISTINCT destination_code, COUNT(flight_no) FROM flights
    WHERE destination_code IN(
        SELECT icao_code FROM airport
    )
    GROUP BY destination_code
'''
cursor.execute(q)

for info in cursor:
    print(info)

('VOCB', 30)
('VOHS', 196)
('VABB', 218)
('VIDP', 190)
('VOBL', 180)
('OMDB', 142)
('VOPB', 6)
('EGLL', 142)
('VOMM', 152)
('VOGO', 40)
('RJTT', 172)
('OIIE', 2)
('LFPG', 120)
('VRMM', 46)
('WMKL', 2)


8. For each airline, count the number of flights by status (e.g., 'On Time', 'Delayed', 'Cancelled') using CASE WHEN.

In [4]:
q8 = '''
SELECT
    airline_code,

    SUM(
        CASE
            WHEN scheduled_departure = actual_departure
            THEN 1
            ELSE 0
        END
    ) AS on_time,

    SUM(
        CASE
            WHEN actual_departure > scheduled_departure
            THEN 1
            ELSE 0
        END
    ) AS delayed_flights,

    SUM(
        CASE
            WHEN actual_departure IS NULL
            THEN 1
            ELSE 0
        END
    ) AS cancelled

FROM flights
GROUP BY airline_code
'''
cursor.execute(q8)

for info in cursor:
    print(info)

('6E', Decimal('691'), Decimal('70'), Decimal('5'))
('AI', Decimal('206'), Decimal('24'), Decimal('0'))
('UL', Decimal('8'), Decimal('0'), Decimal('0'))
('IX', Decimal('92'), Decimal('14'), Decimal('0'))
('EY', Decimal('17'), Decimal('0'), Decimal('1'))
('EK', Decimal('40'), Decimal('62'), Decimal('2'))
('SQ', Decimal('20'), Decimal('4'), Decimal('0'))
('FD', Decimal('6'), Decimal('2'), Decimal('0'))
('AK', Decimal('10'), Decimal('10'), Decimal('0'))
('TR', Decimal('8'), Decimal('0'), Decimal('0'))
('MH', Decimal('8'), Decimal('14'), Decimal('0'))
('CX', Decimal('10'), Decimal('10'), Decimal('0'))
('TG', Decimal('4'), Decimal('8'), Decimal('0'))
('LH', Decimal('10'), Decimal('6'), Decimal('0'))
('KU', Decimal('4'), Decimal('0'), Decimal('0'))
('G9', Decimal('10'), Decimal('2'), Decimal('0'))
('QR', Decimal('6'), Decimal('18'), Decimal('0'))
('3L', Decimal('2'), Decimal('0'), Decimal('0'))
('BA', Decimal('6'), Decimal('44'), Decimal('0'))
('SG', Decimal('24'), Decimal('2'), Decimal('4')

9. Show all cancelled flights, with aircraft and both airports, ordered by departure time descending.

In [8]:
q9 = '''
    SELECT flight_id, aircraft_registration, a1.name AS departure_airport, a2.name AS arrival_airport, scheduled_departure FROM flights
    LEFT JOIN airport a1 ON flights.origin_icao = a1.icao_code
    LEFT JOIN airport a2 ON flights.destination_code = a2.icao_code
    WHERE flights.status = 'Canceled' OR flights.actual_departure IS NULL
    ORDER BY scheduled_departure DESC
'''
cursor.execute(q9)

for info in cursor:
    print(info)

('SG 164_2026-05-20 18:05Z', None, None, 'New Delhi Indira Gandhi', datetime.datetime(2026, 5, 20, 18, 5))
('SG 164_2026-05-20 18:05Z', None, 'Mumbai Chhatrapati Shivaji', 'New Delhi Indira Gandhi', datetime.datetime(2026, 5, 20, 18, 5))
('W5 57_2026-05-20 18:00Z', None, None, None, datetime.datetime(2026, 5, 20, 18, 0))
('W5 57_2026-05-20 18:00Z', None, 'Tehran Imam Khomeini', None, datetime.datetime(2026, 5, 20, 18, 0))
('EK 2027_2026-05-20 16:55Z', None, None, 'Dubai', datetime.datetime(2026, 5, 20, 14, 55))
('FZ 34_2026-05-20 16:55Z', None, None, 'Dubai', datetime.datetime(2026, 5, 20, 14, 55))
('UA 6509_2026-05-20 16:55Z', None, None, 'Dubai', datetime.datetime(2026, 5, 20, 14, 55))
('EK 2027_2026-05-20 16:55Z', None, None, 'Dubai', datetime.datetime(2026, 5, 20, 14, 55))
('UA 6509_2026-05-20 16:55Z', None, None, 'Dubai', datetime.datetime(2026, 5, 20, 14, 55))
('FZ 34_2026-05-20 16:55Z', None, None, 'Dubai', datetime.datetime(2026, 5, 20, 14, 55))
('SG 466_2026-05-20 14:35Z', Non

10. List all city pairs (origin-destination) that have more than 2 different aircraft models operating flights between them.

In [ ]:
q10 = '''
    SELECT a1.city AS departure_city, a2.city AS destination_city, COUNT(DISTINCT aircraft.model) AS model_count FROM flights
    JOIN airport a1 ON flights.origin_icao = a1.icao_code
    JOIN airport a2 ON flights.destination_code = a2.icao_code
    JOIN aircraft ON flights.aircraft_registration = aircraft.registration
    GROUP BY departure_city, destination_city
    HAVING model_count > 2
    ORDER BY model_count DESC
'''
cursor.execute(q10)

for info in cursor:
    print(info)

('Bengaluru', 'Chhatrapati Shivaji', 5)
('Bengaluru', 'Dabolim', 5)
('Chhatrapati Shivaji', 'Bengaluru', 5)
('Dabolim', 'Bengaluru', 5)
('Rajiv Gandhi', 'Bengaluru', 5)
('Bengaluru', 'Chennai', 4)
('Bengaluru', 'Indira Gandhi', 4)
('Chennai', 'Chhatrapati Shivaji', 4)
('Indira Gandhi', 'Bengaluru', 4)
('Bengaluru', 'Rajiv Gandhi', 3)
('Chennai', 'Bengaluru', 3)
('Chennai', 'Indira Gandhi', 3)
('Chhatrapati Shivaji', 'Indira Gandhi', 3)
('Chhatrapati Shivaji', 'Rajiv Gandhi', 3)
('Dubai', 'Chhatrapati Shivaji', 3)
('Dubai', 'Rajiv Gandhi', 3)
('Rajiv Gandhi', 'Chhatrapati Shivaji', 3)


11. For each destination airport, compute the % of delayed flights (status='Delayed') among all arrivals, sorted by highest percentage. 

In [29]:
q11 = '''
    SELECT airport.name AS destination_airport, COUNT(*) AS total_arrivals,
    SUM(
        CASE
            WHEN actual_arrival > scheduled_arrival THEN 1
            ELSE 0
        END
    ) AS delayed_arrivals,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN actual_arrival > scheduled_arrival THEN 1
                ELSE 0
            END
        ) / COUNT(*),2
    ) AS delayed_percentage
    FROM flights
    JOIN airport on flights.destination_code = airport.icao_code
    GROUP BY airport.icao_code, destination_airport
    ORDER BY delayed_percentage DESC
'''
cursor.execute(q11)

for info in cursor:
    print(info)

('Bangalore Bengaluru', 180, Decimal('132'), Decimal('73.33'))
('Tokyo Haneda', 172, Decimal('86'), Decimal('50.00'))
('Paris Charles de Gaulle', 120, Decimal('46'), Decimal('38.33'))
('Dubai', 142, Decimal('28'), Decimal('19.72'))
('London Heathrow', 142, Decimal('16'), Decimal('11.27'))
('Mumbai Chhatrapati Shivaji', 218, Decimal('22'), Decimal('10.09'))
('Coimbatore', 30, Decimal('0'), Decimal('0.00'))
('Hyderabad Rajiv Gandhi', 196, Decimal('0'), Decimal('0.00'))
('New Delhi Indira Gandhi', 190, Decimal('0'), Decimal('0.00'))
('Port Blair Vir Savarkar', 6, Decimal('0'), Decimal('0.00'))
('Chennai', 152, Decimal('0'), Decimal('0.00'))
('Vasco da Gama Dabolim', 40, Decimal('0'), Decimal('0.00'))
('Tehran Imam Khomeini', 2, Decimal('0'), Decimal('0.00'))
('Malé', 46, Decimal('0'), Decimal('0.00'))
('Langkawi', 2, Decimal('0'), Decimal('0.00'))
